In [ ]:
# ============================================================
# 📂 Upload TXT File + Generate 500 Detailed Q&A
# ============================================================
from google.colab import files
import os, json, re, time, math, csv
import google.generativeai as genai

# 🔑 Configure Gemini
API_KEY = " "  # 🔒 Replace with your valid key
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# ============================================================
# 🧹 JSON Cleaning
# ============================================================
def clean_json(raw):
    if not raw:
        return None
    raw = re.sub(r"```json|```", "", raw.strip())
    start, end = raw.find("["), raw.rfind("]") + 1
    if start == -1 or end <= 0:
        return None
    text = raw[start:end]
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    try:
        return json.loads(text)
    except:
        return None

# ============================================================
# 🤖 Generate One Batch of Detailed Q&A
# ============================================================
def generate_detailed_qna(text, seen):
    prompt = f"""
You are an expert educational AI that generates *detailed, high-quality academic Q&A*.

🎯 TASK:
Generate 25 **unique** and **non-repetitive** Question–Answer pairs from the given Hindi or bilingual NCERT text.

Distribute evenly among these 5 types:
1️⃣ Multiple Choice Questions (MCQs)
2️⃣ Objective Questions (True/False, Fill in the Blanks, Match the Following)
3️⃣ Summarization Questions
4️⃣ Chain of Thought Questions
5️⃣ Logical Reasoning Questions

⚙️ OUTPUT RULES:
- Return a **valid JSON array only**, no markdown, no text.
- Each object must follow one of these schemas:

🅰️ For MCQs:
{{
  "type": "MCQ",
  "question": "string (the question in Hindi or bilingual)",
  "options": ["Option A", "Option B", "Option C", "Option D"],
  "correct_answer": "string (exactly one of the options)",
  "explanation": "detailed explanation (3–6 sentences)"
}}

📘 For other types:
{{
  "type": "Objective" | "Summarization" | "Chain of Thought" | "Logical Reasoning",
  "question": "string",
  "answer": "long, detailed, well-explained answer (4–8 sentences)"
}}

⚠️ RULES:
- Avoid exact duplicate questions.
- Semantically similar questions are allowed if answers differ in explanation.
- Maintain conceptual and difficulty variety.
- Output clean JSON only.

Text:
{text[:7000]}
"""

    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                response_mime_type="application/json",
                temperature=0.95,
                top_p=0.95,
                top_k=40,
            ),
        )
        return clean_json(response.text)
    except Exception as e:
        print("⚠️ Gemini error:", e)
        return None

# ============================================================
# 🚀 Main Function – Generate 500 Detailed Q&A
# ============================================================
print("📁 Please upload your .txt file (e.g. Chapter_2.txt).")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

TOTAL_Q = 500
BATCH_SIZE = 25
CHUNK_SIZE = 6000  # to prevent overload
MAX_RETRIES = 3

BASE_NAME = os.path.splitext(filename)[0]
OUTPUT_JSON = f"{BASE_NAME}_500_detailed_qna.json"
OUTPUT_CSV = f"{BASE_NAME}_500_detailed_qna.csv"

all_qna = []
seen = set()   # tracks *exact* duplicates only
num_batches = math.ceil(TOTAL_Q / BATCH_SIZE)

print(f"\n🧠 Generating {TOTAL_Q} detailed and diverse questions in {num_batches} batches...\n")

for batch in range(1, num_batches + 1):
    print(f"\n⚙️ Batch {batch}/{num_batches} ...")

    # 🧩 Step 1: Chunk text dynamically
    start = ((batch - 1) * CHUNK_SIZE) % len(text)
    chunk_text = text[start:start + CHUNK_SIZE]

    # 🧩 Step 2: Retry logic
    qna_data = None
    for attempt in range(1, MAX_RETRIES + 1):
        qna_data = generate_detailed_qna(chunk_text, seen)
        if qna_data:
            break
        print(f"⚠️ Retry {attempt}/{MAX_RETRIES} for batch {batch} ...")
        time.sleep(5)

    if not qna_data:
        print(f"❌ Skipping batch {batch} after {MAX_RETRIES} failed attempts.")
        continue

    # 🧩 Step 3: Filter duplicates (exact only)
    new_items = []
    for item in qna_data:
        q = item.get("question", "").strip()
        if not q:
            continue
        q_norm = re.sub(r'\s+', ' ', q.lower())
        if q_norm not in seen:
            seen.add(q_norm)
            new_items.append(item)

    all_qna.extend(new_items)
    print(f"✅ Added {len(new_items)} new (Total: {len(all_qna)}/{TOTAL_Q})")

    # 🧩 Step 4: Save progress incrementally
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_qna, f, ensure_ascii=False, indent=2)

    if len(all_qna) >= TOTAL_Q:
        break

    time.sleep(3)

# ============================================================
# 💾 Final Save – JSON & CSV
# ============================================================
# Save final JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_qna[:TOTAL_Q], f, ensure_ascii=False, indent=2)

# Save CSV file
with open(OUTPUT_CSV, "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Type", "Question", "Options", "Correct Answer", "Explanation / Answer"])
    for q in all_qna[:TOTAL_Q]:
        qtype = q.get("type", "")
        question = q.get("question", "")
        options = ", ".join(q.get("options", [])) if "options" in q else ""
        correct = q.get("correct_answer", "")
        explanation = q.get("explanation", q.get("answer", ""))
        writer.writerow([qtype, question, options, correct, explanation])

print(f"\n🎉 Done! {len(all_qna)} detailed questions saved as:")
print(f"📘 JSON → {OUTPUT_JSON}")
print(f"📗 CSV  → {OUTPUT_CSV}")

files.download(OUTPUT_JSON)
files.download(OUTPUT_CSV)

📁 Please upload your .txt file (e.g. Chapter_2.txt).


Saving khph1an.txt to khph1an.txt

🧠 Generating 500 detailed and diverse questions in 20 batches...


⚙️ Batch 1/20 ...
⚠️ Retry 1/3 for batch 1 ...
⚠️ Retry 2/3 for batch 1 ...
⚠️ Retry 3/3 for batch 1 ...
❌ Skipping batch 1 after 3 failed attempts.

⚙️ Batch 2/20 ...
⚠️ Retry 1/3 for batch 2 ...
